# Swin Transformer Tutorial

This notebook covers three things:

1. A visual comparison of ViT patching vs. Swin window patching.
2. A from-scratch Swin Transformer implementation inspired by the article below, with tensor shapes printed stage by stage.
3. A Hugging Face Transformers benchmark comparing ViT and Swin on images of multiple resolutions.

Reference material:
- Article: [Swin-Transformer from Scratch in PyTorch](https://python.plainenglish.io/swin-transformer-from-scratch-in-pytorch-31275152bf03)
- Hugging Face Swin docs: https://huggingface.co/docs/transformers/model_doc/swin
- Hugging Face ViT docs: https://huggingface.co/docs/transformers/model_doc/vit


In [ ]:
import math
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from matplotlib.patches import Rectangle
from torch.nn import functional as F

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Change these placeholders to local files on your machine.
IMAGE_PATH = r'C:\path\to\your\image.jpg'
IMAGE_PATHS = [
    r'C:\path\to\your\image_224.jpg',
    r'C:\path\to\your\image_384.jpg',
    r'C:\path\to\your\image_512.jpg',
]

# Section 1 uses a 128x128 canvas so patch_size=4 produces a 32x32 token grid.
VIS_IMAGE_SIZE = 128
VIS_PATCH_SIZE = 4
VIS_WINDOW_SIZE = 16

PATCH_SIZE = 16
WINDOW_SIZE = 7

def print_shape(label, tensor):
    print(f'{label:<40} {tuple(tensor.shape)}')

def load_image(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Image path does not exist: {path}')
    return Image.open(path).convert('RGB')

def resize_square(image, size):
    resample = getattr(Image, 'Resampling', Image).BICUBIC
    return image.resize((size, size), resample=resample)

def draw_grid(ax, width, height, step, color, alpha=0.75, linewidth=1.0):
    for x in range(0, width, step):
        ax.axvline(x, color=color, alpha=alpha, linewidth=linewidth)
    for y in range(0, height, step):
        ax.axhline(y, color=color, alpha=alpha, linewidth=linewidth)
    ax.add_patch(Rectangle((0, 0), width, height, fill=False, edgecolor=color, linewidth=2))

def draw_window_overlay(ax, width, height, patch_size, window_size):
    window_pixels = patch_size * window_size
    colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange']
    window_index = 0
    for top in range(0, height, window_pixels):
        for left in range(0, width, window_pixels):
            color = colors[window_index % len(colors)]
            ax.add_patch(
                Rectangle(
                    (left, top),
                    window_pixels,
                    window_pixels,
                    facecolor=color,
                    edgecolor=color,
                    linewidth=2.5,
                    alpha=0.18,
                )
            )
            ax.text(
                left + 4,
                top + 14,
                f'W{window_index + 1}',
                color=color,
                fontsize=12,
                weight='bold',
                bbox=dict(facecolor='white', alpha=0.55, edgecolor='none', pad=1.5),
            )
            window_index += 1
    draw_grid(ax, width, height, patch_size, color='white', alpha=0.30, linewidth=0.4)

def show_patching_comparison(image_path, vis_image_size, patch_size, window_size):
    image = resize_square(load_image(image_path), vis_image_size)
    width, height = image.size
    token_grid = width // patch_size
    window_tokens = window_size

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    axes[0].imshow(image)
    axes[0].set_title(f'ViT: {token_grid}x{token_grid} tokens from patch size {patch_size}')
    axes[0].axis('off')
    draw_grid(axes[0], width, height, patch_size, color='cyan')

    axes[1].imshow(image)
    axes[1].set_title(
        f'Swin: {token_grid}x{token_grid} tokens with {token_grid // window_tokens}x{token_grid // window_tokens} windows of {window_tokens}x{window_tokens} tokens'
    )
    axes[1].axis('off')
    draw_window_overlay(axes[1], width, height, patch_size, window_size)

    plt.tight_layout()
    plt.show()
    return image


## 1. ViT Patchification vs. Swin Windows

Use `VIS_PATCH_SIZE = 4` and `VIS_WINDOW_SIZE = 16` for the overlay below. The image is resized to a 128x128 canvas so the patch grid becomes 32x32 tokens, and the Swin view shows four colored 16x16-token windows on the same grid.

Replace `IMAGE_PATH` with a local image path from your machine.


In [ ]:
# Replace IMAGE_PATH with your local file before running.
_ = show_patching_comparison(IMAGE_PATH, VIS_IMAGE_SIZE, VIS_PATCH_SIZE, VIS_WINDOW_SIZE)


## 2. Swin Transformer From Scratch

The implementation below follows the same high-level flow as the article: patch embedding, window attention with optional shifting, patch merging, and a classification head.

The model is intentionally small enough to be readable, but the tensor flow matches the real Swin pattern.


In [ ]:
def window_partition(x, window_size):
    batch_size, height, width, channels = x.shape
    x = x.view(batch_size, height // window_size, window_size, width // window_size, window_size, channels)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size * window_size, channels)
    return windows

def window_reverse(windows, window_size, height, width, batch_size):
    channels = windows.shape[-1]
    x = windows.view(batch_size, height // window_size, width // window_size, window_size, window_size, channels)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(batch_size, height, width, channels)
    return x

class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        print_shape('input', x)
        x = self.proj(x)
        print_shape('patch_embed.conv', x)
        x = x.permute(0, 2, 3, 1).contiguous()
        print_shape('patch_embed.permute', x)
        x = self.norm(x)
        print_shape('patch_embed.norm', x)
        return x

class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        coords = torch.stack(torch.meshgrid(torch.arange(window_size), torch.arange(window_size), indexing='ij'))
        coords_flatten = coords.reshape(2, -1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer('relative_position_index', relative_position_index)

    def forward(self, x, attn_mask=None):
        batch_windows, token_count, channels = x.shape
        qkv = self.qkv(x).reshape(batch_windows, token_count, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale

        relative_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)]
        relative_bias = relative_bias.view(token_count, token_count, -1).permute(2, 0, 1)
        attn = attn + relative_bias.unsqueeze(0)

        if attn_mask is not None:
            num_windows = attn_mask.shape[0]
            attn = attn.view(batch_windows // num_windows, num_windows, self.num_heads, token_count, token_count)
            attn = attn + attn_mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, token_count, token_count)

        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(batch_windows, token_count, channels)
        x = self.proj(x)
        return x

class SwinBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio)

        if shift_size > 0:
            height, width = input_resolution
            img_mask = torch.zeros((1, height, width, 1))
            h_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
            w_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
            count = 0
            for h_slice in h_slices:
                for w_slice in w_slices:
                    img_mask[:, h_slice, w_slice, :] = count
                    count += 1
            mask_windows = window_partition(img_mask, window_size)
            mask_windows = mask_windows.view(-1, window_size * window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float('-inf')).masked_fill(attn_mask == 0, 0.0)
        else:
            attn_mask = None
        self.register_buffer('attn_mask', attn_mask)

    def forward(self, x):
        batch_size, height, width, channels = x.shape
        shortcut = x
        x = self.norm1(x)
        print_shape('  block.norm1', x)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            print_shape('  block.shift', x)

        windows = window_partition(x, self.window_size)
        print_shape('  block.window_partition', windows)
        windows = self.attn(windows, self.attn_mask)
        print_shape('  block.attention', windows)
        x = window_reverse(windows, self.window_size, height, width, batch_size)
        print_shape('  block.window_reverse', x)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
            print_shape('  block.reverse_shift', x)

        x = shortcut + x
        print_shape('  block.residual1', x)
        x = x + self.mlp(self.norm2(x))
        print_shape('  block.residual2', x)
        return x

class PatchMerging(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.norm = nn.LayerNorm(4 * input_dim)
        self.reduction = nn.Linear(4 * input_dim, 2 * input_dim, bias=False)

    def forward(self, x):
        batch_size, height, width, channels = x.shape
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        print_shape('  patch_merging.concat', x)
        x = self.norm(x)
        print_shape('  patch_merging.norm', x)
        x = self.reduction(x)
        print_shape('  patch_merging.reduction', x)
        return x

class TinySwinTransformer(nn.Module):
    def __init__(self, image_size=224, patch_size=4, window_size=7, num_classes=1000, embed_dim=96, depths=(2, 2, 2, 2), num_heads=(3, 6, 12, 24)):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size=patch_size, embed_dim=embed_dim)
        self.stages = nn.ModuleList()
        current_dim = embed_dim
        current_resolution = image_size // patch_size

        for stage_index, depth in enumerate(depths):
            blocks = []
            for block_index in range(depth):
                shift_size = 0 if block_index % 2 == 0 else window_size // 2
                blocks.append(
                    SwinBlock(
                        dim=current_dim,
                        input_resolution=(current_resolution, current_resolution),
                        num_heads=num_heads[stage_index],
                        window_size=window_size,
                        shift_size=shift_size,
                    )
                )
            downsample = PatchMerging(current_dim) if stage_index < len(depths) - 1 else None
            self.stages.append(nn.ModuleDict({'blocks': nn.ModuleList(blocks), 'downsample': downsample}))
            if downsample is not None:
                current_resolution //= 2
                current_dim *= 2

        self.norm = nn.LayerNorm(current_dim)
        self.head = nn.Linear(current_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        print_shape('after_patch_embed', x)

        for stage_index, stage in enumerate(self.stages, start=1):
            print(f'\nStage {stage_index}')
            for block_index, block in enumerate(stage['blocks'], start=1):
                print(f' Block {block_index}')
                x = block(x)
                print_shape(f'  stage{stage_index}.block{block_index}.output', x)
            if stage['downsample'] is not None:
                print(' Downsample')
                x = stage['downsample'](x)
                print_shape(f'  stage{stage_index}.downsample.output', x)

        x = self.norm(x)
        print_shape('final_norm', x)
        x = x.mean(dim=(1, 2))
        print_shape('global_average_pool', x)
        logits = self.head(x)
        print_shape('logits', logits)
        return logits


In [ ]:
dummy = torch.randn(1, 3, 224, 224)
scratch_swin = TinySwinTransformer(image_size=224, patch_size=4, window_size=WINDOW_SIZE, num_classes=1000).to(device)

with torch.inference_mode():
    logits = scratch_swin(dummy.to(device))

print('Scratch Swin output shape:', tuple(logits.shape))


## 3. Attention Benchmark: ViT vs. Swin

This section compares the attention operation time for a vanilla ViT attention block and a Swin window attention block.

- No pretrained checkpoints are used.
- The images are not resized.
- Only minimal right/bottom padding is applied when needed so patch and window partitioning stay valid.

The benchmark uses `patch_size = 16` and `window_size = 4` patch tokens, so each Swin window covers `4 x 4` patches.

Replace the paths in `IMAGE_PATHS` with images from your system before running this section.


In [ ]:
ATTN_PATCH_SIZE = 16
ATTN_WINDOW_SIZE = 4
ATTN_EMBED_DIM = 96
ATTN_NUM_HEADS = 3
ATTN_RUNS = 20
ATTN_WARMUP = 5

def load_image_tensor(image_path):
    image = load_image(image_path)
    array = np.array(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return tensor

def pad_to_multiple(tensor, multiple):
    _, _, height, width = tensor.shape
    pad_height = (multiple - height % multiple) % multiple
    pad_width = (multiple - width % multiple) % multiple
    if pad_height == 0 and pad_width == 0:
        return tensor
    return F.pad(tensor, (0, pad_width, 0, pad_height))

class PatchEmbed(nn.Module):
    def __init__(self, in_channels=3, embed_dim=96, patch_size=16):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        print_shape('patch_embed.output', x)
        return x

class VanillaViTAttention(nn.Module):
    def __init__(self, embed_dim=96, num_heads=3):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

    def forward(self, tokens, trace=False):
        tokens = self.norm(tokens)
        if trace:
            print_shape('vit.norm', tokens)
        attn_output, _ = self.attn(tokens, tokens, tokens, need_weights=False)
        if trace:
            print_shape('vit.attention', attn_output)
        return attn_output

class SwinWindowAttention(nn.Module):
    def __init__(self, embed_dim=96, num_heads=3, window_size=4):
        super().__init__()
        self.window_size = window_size
        self.norm = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

    def window_partition(self, x):
        batch_size, height, width, channels = x.shape
        window_size = self.window_size
        x = x.view(batch_size, height // window_size, window_size, width // window_size, window_size, channels)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size * window_size, channels)
        return x

    def window_reverse(self, windows, height, width, batch_size):
        window_size = self.window_size
        channels = windows.shape[-1]
        x = windows.view(batch_size, height // window_size, width // window_size, window_size, window_size, channels)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(batch_size, height, width, channels)
        return x

    def forward(self, feature_map, trace=False):
        feature_map = self.norm(feature_map)
        if trace:
            print_shape('swin.norm', feature_map)

        batch_size, height, width, channels = feature_map.shape
        windows = self.window_partition(feature_map)
        if trace:
            print_shape('swin.window_partition', windows)

        windows, _ = self.attn(windows, windows, windows, need_weights=False)
        if trace:
            print_shape('swin.window_attention', windows)

        feature_map = self.window_reverse(windows, height, width, batch_size)
        if trace:
            print_shape('swin.window_reverse', feature_map)
        return feature_map

def benchmark_module(module, inputs, runs=20, warmup=5):
    module.eval()
    with torch.inference_mode():
        for _ in range(warmup):
            _ = module(inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        output = None
        for _ in range(runs):
            output = module(inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
    return (end - start) / runs, output


In [ ]:
patch_embed = PatchEmbed(embed_dim=ATTN_EMBED_DIM, patch_size=ATTN_PATCH_SIZE).to(device).eval()
vit_attention = VanillaViTAttention(embed_dim=ATTN_EMBED_DIM, num_heads=ATTN_NUM_HEADS).to(device).eval()
swin_attention = SwinWindowAttention(embed_dim=ATTN_EMBED_DIM, num_heads=ATTN_NUM_HEADS, window_size=ATTN_WINDOW_SIZE).to(device).eval()

pad_multiple = ATTN_PATCH_SIZE * ATTN_WINDOW_SIZE

for image_path in IMAGE_PATHS:
    image = load_image(image_path)
    image_tensor = load_image_tensor(image_path).to(device)
    padded_tensor = pad_to_multiple(image_tensor, pad_multiple)

    with torch.inference_mode():
        features = patch_embed(padded_tensor)
        vit_tokens = features.flatten(2).transpose(1, 2).contiguous()
        swin_map = features.permute(0, 2, 3, 1).contiguous()

    vit_time, vit_output = benchmark_module(vit_attention, vit_tokens, runs=ATTN_RUNS, warmup=ATTN_WARMUP)
    swin_time, swin_output = benchmark_module(swin_attention, swin_map, runs=ATTN_RUNS, warmup=ATTN_WARMUP)

    print('\n' + '=' * 90)
    print(f'Image: {image_path}')
    print(f'Original size: {image.size} | Padded size: {tuple(padded_tensor.shape[-2:][::-1])}')
    print_shape('ViT token tensor', vit_tokens)
    print_shape('Swin feature map', swin_map)
    print(f'ViT attention time:  {vit_time:.6f} s/run')
    print(f'Swin attention time: {swin_time:.6f} s/run')
    print(f'Difference (ViT - Swin): {vit_time - swin_time:+.6f} s/run')
    print_shape('ViT output', vit_output)
    print_shape('Swin output', swin_output)
